# Assess How Well Top-Ranked Ligands Can Predict a Gene Set of Interest

This notebook assesses the ligands prioritized by NicheNet in their ability to predict a gene set of interest. We first run a NicheNet analysis to obtain ligand rankings, then evaluate the top 30 ligands via cross-validated random forest classification, fraction-of-top-predicted analysis, and Fisher's exact test.

In [ ]:
import os
os.environ.setdefault("NICHENETR_DATA_DIR", "path/to/nichenetr_data")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import reduce

import nichenetr as nn

## Run NicheNet

We run the NicheNet wrapper to obtain ligand rankings.

In [ ]:
lr_network = nn.load_lr_network("mouse")
ligand_target_matrix = nn.load_ligand_target_matrix("mouse")
weighted_networks = nn.load_weighted_networks("mouse")

lr_network = lr_network[["from", "to"]].drop_duplicates()

adata = nn.load_seurat_obj()
adata = nn.alias_to_symbol_anndata(adata, "mouse")

In [ ]:
nichenet_output = nn.nichenet_seuratobj_aggregate(
    receiver="CD8 T",
    adata=adata,
    sender=["CD4 T", "Treg", "Mono", "NK", "B", "DC"],
    condition_col="aggregate",
    condition_oi="LCMV",
    condition_ref="SS",
    celltype_col="celltype",
    expression_pct=0.05,
    ligand_target_matrix=ligand_target_matrix,
    lr_network=lr_network,
    weighted_networks=weighted_networks,
)

best_upstream_ligands = (
    nichenet_output["ligand_activities"]
    .nlargest(30, "aupr_corrected")["test_ligand"]
    .tolist()
)
print(f"Top 30 ligands: {best_upstream_ligands}")

## Assess how well top-ranked ligands can predict a gene set of interest

For the top 30 ligands, we build a multi-ligand model using cross-validated random forest classification. The model predicts whether a gene belongs to the gene set of interest.

In [ ]:
# Cross-validation settings: 3-fold, 2 rounds
k = 3
n = 2

gene_predictions_top30_list = [
    nn.assess_rf_class_probabilities(
        round_num=i,
        folds=k,
        geneset=nichenet_output["geneset_oi"],
        background_expressed_genes=nichenet_output["background_expressed_genes"],
        ligands_oi=best_upstream_ligands,
        ligand_target_matrix=ligand_target_matrix,
    )
    for i in range(1, n + 1)
]

### Evaluate classification performance

Compute AUROC, AUPR, and Pearson correlation for each cross-validation round.

In [ ]:
target_prediction_performances_cv = pd.concat(
    [
        nn.classification_evaluation_continuous_pred_wrapper(df).assign(round=i + 1)
        for i, df in enumerate(gene_predictions_top30_list)
    ],
    ignore_index=True,
)
target_prediction_performances_cv

In [ ]:
print(f"Mean AUROC: {target_prediction_performances_cv['auroc'].mean():.4f}")
print(f"Mean AUPR: {target_prediction_performances_cv['aupr'].mean():.4f}")
print(f"Mean Pearson: {target_prediction_performances_cv['pearson'].mean():.4f}")

### Fraction of top-predicted targets

Evaluate whether genes in the gene set are more likely to be among the top 5% predicted targets.

In [ ]:
target_prediction_performances_discrete_cv = pd.concat(
    [
        nn.calculate_fraction_top_predicted(
            round_num=i + 1,
            response_prediction_df=df,
            ligands_oi=best_upstream_ligands,
            ligand_target_matrix=ligand_target_matrix,
            quantile_cutoff=0.95,
        ).assign(round=i + 1)
        for i, df in enumerate(gene_predictions_top30_list)
    ],
    ignore_index=True,
)
target_prediction_performances_discrete_cv

In [ ]:
# Fraction of true targets among top 5% predicted
frac_true = (
    target_prediction_performances_discrete_cv[
        target_prediction_performances_discrete_cv["true_target"] == True
    ]["fraction_positive_predicted"].mean()
)
print(f"Fraction of gene-set genes in top 5% predicted: {frac_true:.4f}")

frac_false = (
    target_prediction_performances_discrete_cv[
        target_prediction_performances_discrete_cv["true_target"] == False
    ]["fraction_positive_predicted"].mean()
)
print(f"Fraction of non-gene-set genes in top 5% predicted: {frac_false:.4f}")

### Fisher's exact test

Test whether gene-set members are significantly enriched in the top-predicted targets.

In [ ]:
fisher_pvalues = [
    nn.calculate_fraction_top_predicted_fisher(
        round_num=i + 1,
        response_prediction_df=df,
        ligands_oi=best_upstream_ligands,
        ligand_target_matrix=ligand_target_matrix,
        quantile_cutoff=0.95,
    )
    for i, df in enumerate(gene_predictions_top30_list)
]

print(f"Fisher p-values: {fisher_pvalues}")
print(f"Average Fisher p-value: {np.mean(fisher_pvalues):.6f}")

### Get top-predicted genes

Look at which genes are well-predicted in every cross-validation round.

In [ ]:
top_predicted_genes_list = [
    nn.get_top_predicted_genes(
        round_num=i + 1,
        response_prediction_df=gene_predictions_top30_list[i],
        ligands_oi=best_upstream_ligands,
        ligand_target_matrix=ligand_target_matrix,
    )
    for i in range(n)
]

# Combine across rounds
top_predicted_genes = reduce(
    lambda left, right: pd.merge(left, right, on=["gene", "true_target"], how="outer"),
    top_predicted_genes_list,
)

print("Top predicted true target genes:")
top_predicted_genes[top_predicted_genes["true_target"] == True]